# Development 5c: predict every scene of one block with VGGT, the predecessor (GPU)

The comparison arm. Same scenes, same camera, same frames as notebook `05a_one_block_predict`, run
through `facebook/VGGT-1B` (CC BY-NC 4.0). Nothing is scored here.

**Needs a GPU server.** The first run also downloads a 5 GB checkpoint. VGGT
resizes 1920x1200 to 518x322, where VGGT-Omega uses 640x400. Scenes already
predicted are skipped.

In [1]:
# --- 1. Configuration ---
PERSIST_MODE = "drive"
SPLIT, BLOCK = "val", 11
LAYERS = ["camera_keyframes"]
CAMERA = "front_medium"

In [ ]:
# === CODE SYNC (auto-generated by `python -m vggt_aura.sync`, do not edit) ===
raise RuntimeError("The sync cell is empty. On your own machine, in the project folder, run:  python -m vggt_aura.sync   and reopen this notebook.")

In [3]:
# --- 3. Start the session, then add the VGGT package (only this arm needs it) ---
from vggt_aura import session as sess

session = sess.start_session(persist_mode=PERSIST_MODE, require_gpu=False)
sess.install_extra("vggt", sess.VGGT_PIP_ARGS)

Mounted at /content/drive
installing vggt_omega
installing pybind11
installing fzi_aura
persist root: /content/drive/MyDrive/vggt-omega-aura-benchmark
data root   : /content/data/fzi-aura (runtime disk, wiped at session end)
installing vggt


In [4]:
# --- 4. Make sure the camera layer of the block is on this runtime ---
from vggt_aura import aura_data as ad, inference as inf, pins

chunks, scene_blocks, hub_files = ad.fetch_release_tables(session.data_root / "_release_tables")
BLOCK_SCENE_IDS = ad.block_scene_ids(scene_blocks, SPLIT, BLOCK)
if ad.block_on_disk(session.data_root, BLOCK_SCENE_IDS, LAYERS):
    print("block already on disk")
else:
    ad.download_block(session.data_root, SPLIT, BLOCK, BLOCK_SCENE_IDS, LAYERS)

$ /usr/bin/python3 -m fzi_aura.download /content/data/fzi-aura --revision 3404bd6b8fcd6eed53a0ec7610650a6393aabb49 --splits val --scene-ids-file /content/data/fzi-aura/_scene_ids_val_block000011.txt --layers camera_keyframes --jobs 8 --verify


In [5]:
# --- 5. Predict every scene not yet done ---
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from fzi_aura import FZIAURADataset

MODEL_NAME = inf.VGGT_NAME
STAGE = f"predict/{CAMERA}/{MODEL_NAME}"
dataset = FZIAURADataset(session.data_root, split=SPLIT)
model, log = None, []

for scene_id in BLOCK_SCENE_IDS:
    scene = dataset.get_scene(scene_id)
    npz_path, json_path = inf.prediction_paths(session.persist_root, scene.name, CAMERA, MODEL_NAME)
    if session.manifest.is_done(scene_id, STAGE) and npz_path.is_file():
        log.append({"scene_id": scene_id, "status": "already done"})
        continue
    frames = inf.select_frames(scene, CAMERA)
    if len(frames) < 3:
        log.append({"scene_id": scene_id, "status": f"skipped: only {len(frames)} frames"})
        continue
    if model is None:
        import torch
        if not torch.cuda.is_available():
            raise RuntimeError("There are scenes left to predict and the model needs a GPU. Connect to a GPU server and re-run.")
        started = time.time()
        model = inf.load_vggt()
        print(f"VGGT ready in {time.time() - started:.0f} s on {torch.cuda.get_device_name(0)}")
    arrays = inf.run_vggt(model, [frame.camera_path(CAMERA) for frame in frames])
    arrays["gt_camera0_from_camera"] = inf.ground_truth_cameras(frames, CAMERA)
    arrays["timestamps_ns"] = np.array([frame.timestamp_ns for frame in frames], dtype=np.int64)
    meta = {"scene_id": scene_id, "scene_name": scene.name, "camera": CAMERA, "frames": len(frames),
            "model": MODEL_NAME, "input_hw": arrays["input_hw"].tolist(),
            "dataset_revision": pins.AURA_DATASET_REVISION, "model_code_commit": pins.VGGT_COMMIT,
            "model_weights_revision": pins.VGGT_HF_REVISION, "gpu": torch.cuda.get_device_name(0),
            "forward_seconds": round(arrays["seconds"], 2), "peak_gpu_gb": round(arrays["peak_gpu_gb"], 2),
            "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds")}
    size_mb = inf.save_predictions(npz_path, json_path, arrays, meta)
    session.manifest.mark_done(scene_id, STAGE, file=str(npz_path.relative_to(session.persist_root)))
    log.append({"scene_id": scene_id, "status": "predicted", "frames": len(frames), "input_wxh": f"{arrays['input_hw'][1]}x{arrays['input_hw'][0]}",
                "seconds": meta["forward_seconds"], "peak_gpu_gb": meta["peak_gpu_gb"], "saved_mb": round(size_mb, 1)})

print(pd.DataFrame(log).to_string(index=False))
print()
print("scenes with VGGT predictions on Drive:", len(session.manifest.done(STAGE)), "of", len(BLOCK_SCENE_IDS))
print("You can remove the GPU server now. Notebook `05d_one_block_metrics_vggt` runs on a CPU.")

VGGT ready in 28 s on NVIDIA A100-SXM4-40GB


/usr/local/lib/python3.13/dist-packages/vggt/models/vggt.py:65: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):


               scene_id    status  frames input_wxh  seconds  peak_gpu_gb  saved_mb
 2025-07-15-13-11-47|24 predicted      40   518x322     3.02         9.75      30.1
 2025-06-11-13-47-25|25 predicted      40   518x322     1.77         9.75      28.8
 2025-06-13-07-09-37|78 predicted      40   518x322     1.76         9.75      28.6
 2025-07-15-14-15-47|71 predicted      40   518x322     1.78         9.75      30.5
 2025-07-15-13-11-47|63 predicted      40   518x322     1.79         9.75      30.7
2025-06-11-12-27-55|207 predicted      40   518x322     1.76         9.75      26.7
2025-07-15-13-11-47|110 predicted      40   518x322     1.77         9.75      30.0
 2025-06-16-12-35-26|82 predicted      39   518x322     1.71         9.70      28.9
  2025-06-16-12-35-26|2 predicted      40   518x322     1.75         9.75      30.3
 2025-07-15-13-11-47|10 predicted      40   518x322     1.78         9.75      30.3
 2025-07-15-13-11-47|48 predicted      40   518x322     1.76         9.75   